# cloudposterior: Monitoring

Monitor MCMC sampling remotely with a live dashboard or push notifications.

- **`dashboard=True`** (default for remote) -- live web dashboard with convergence diagnostics, trace plots, and a stop button. Open on your phone or any browser via QR code.
- **`notify=True`** -- push notifications via [ntfy](https://ntfy.sh) when sampling starts and completes.

> Run this notebook locally to see the QR codes and links.

In [4]:
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az

import cloudposterior as cp

In [5]:
df = pd.read_csv(pm.get_data("radon.csv"))

with pm.Model(name="radon_intercepts", coords={"county": df.county.unique()}) as radon:
    mu_a = pm.Normal("mu_a", mu=0, sigma=5)
    sigma_a = pm.HalfNormal("sigma_a", sigma=2)
    a_raw = pm.Normal("a_raw", mu=0, sigma=1, dims="county")
    a = pm.Deterministic("a", mu_a + sigma_a * a_raw, dims="county")
    b_floor = pm.Normal("b_floor", mu=0, sigma=5)
    mu = a[df.county_code.values] + b_floor * df.floor.values
    sigma_y = pm.HalfNormal("sigma_y", sigma=2)
    pm.Normal("obs", mu=mu, sigma=sigma_y, observed=df.log_radon.values)

## Live dashboard

The dashboard is on by default for remote runs. Scan the QR code or open the URL to see:

- Per-chain progress bars with speed, divergences, and ETA
- Live convergence diagnostics (R-hat, ESS) with color-coded status
- Live trace plots and posterior KDE per parameter
- A stop button to end sampling early if convergence looks good

In [ ]:
with cp.cloud(radon, remote=True, cache=False):
    idata = pm.sample(draws=10000, tune=1000, chains=4)

In [ ]:
az.summary(idata, filter_vars="like", var_names=["mu_a", "sigma_a", "b_floor", "sigma_y"])

In [ ]:
az.plot_trace(idata, filter_vars="like", var_names=["mu_a", "sigma_a", "b_floor", "sigma_y"]);

## Push notifications

Pass `notify=True` to get push notifications when sampling starts and completes. Works for both local and remote runs. Scan the QR code with the [ntfy app](https://ntfy.sh) to subscribe.

For private notifications, point to your own [ntfy server](https://docs.ntfy.sh/install/) with `notify={"server": "https://ntfy.example.com"}`.

In [ ]:
with cp.cloud(radon, cache=False, notify=True):
    idata = pm.sample(draws=2000, tune=1000, chains=4)

## Both

Use both together to get the live dashboard AND push notifications.

In [ ]:
with cp.cloud(radon, remote=True, cache=False, notify=True):
    idata = pm.sample(draws=1000, tune=1000, chains=4)